<a href="https://colab.research.google.com/github/henriquekurata/TC_FASE_4/blob/main/FASE_4_OBESITY_Notebook_Principal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Desenvolvimento do modelo de Machine Learning

Neste projeto, foi desenvolvido um modelo de Machine Learning com o objetivo de auxiliar a equipe médica na previsão do nível de obesidade de pacientes, utilizando dados demográficos, físicos e comportamentais.

Inicialmente, foi realizada a importação das bibliotecas necessárias, incluindo ferramentas para manipulação de dados, visualização, pré-processamento, modelagem e avaliação. Essas bibliotecas permitiram estruturar todo o fluxo de ciência de dados de forma organizada e reproduzível.

Em seguida, o conjunto de dados Obesity.csv foi carregado e analisado de forma exploratória. Essa etapa teve como objetivo compreender a estrutura dos dados, identificar os tipos das variáveis e confirmar que a coluna Obesity representa o alvo do problema. Observou-se que o problema se trata de uma classificação multiclasse, pois o nível de obesidade é categorizado em diferentes classes.

Após essa análise inicial, os dados foram separados em variáveis explicativas (X) e variável alvo (y). As variáveis explicativas foram divididas em numéricas e categóricas, uma vez que cada tipo exige um tratamento diferente durante o pré-processamento. As variáveis numéricas, como idade, peso, altura e nível de atividade física, foram padronizadas para garantir que todas estivessem na mesma escala. Já as variáveis categóricas, como gênero, hábitos alimentares e meio de transporte, foram transformadas por meio de codificação One-Hot, permitindo que fossem utilizadas corretamente pelos algoritmos de Machine Learning.

Todo esse processo de transformação foi encapsulado em um ColumnTransformer, garantindo que o mesmo pré-processamento fosse aplicado tanto durante o treinamento quanto na fase de predição de novos dados. Em seguida, foi criado um Pipeline de Machine Learning, integrando o pré-processamento e o modelo em uma única estrutura, o que assegura consistência, reprodutibilidade e facilidade de deploy.

Na etapa de modelagem, o conjunto de dados foi dividido em dados de treino e teste, utilizando uma estratégia estratificada para manter a proporção das classes de obesidade. Três algoritmos de Machine Learning foram treinados e avaliados: Regressão Logística, utilizada como modelo base; Random Forest, por sua robustez; e Gradient Boosting, conhecido por sua alta capacidade de generalização.

Os modelos foram comparados utilizando a métrica de acurácia, além de relatórios de classificação. Após a avaliação, o modelo Gradient Boosting Classifier apresentou o melhor desempenho, alcançando uma acurácia superior a 75%, valor exigido pelo desafio, e superando os demais algoritmos testados.

Com base nesses resultados, o modelo Gradient Boosting foi selecionado como o modelo final do sistema preditivo. Esse modelo foi então salvo em um arquivo (modelo_obesidade.pkl), contendo tanto o pré-processamento quanto o algoritmo treinado. Dessa forma, o modelo pode ser reutilizado diretamente em uma aplicação web sem a necessidade de novo treinamento.

In [ ]:
!pip install scikit-learn==1.3.2 joblib==1.3.2 xgboost==2.0.3

import pandas as pd
import numpy as np

# Pré-processamento e ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

# Avaliação
from sklearn.metrics import accuracy_score

# Salvar modelo
import joblib


In [ ]:
df = pd.read_csv("Obesity.csv")
df.head()

,Gender,Age,Height,Weight,family_history,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,Obesity
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2111 entries, 0 to 2110
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Gender          2111 non-null   object 
 1   Age             2111 non-null   float64
 2   Height          2111 non-null   float64
 3   Weight          2111 non-null   float64
 4   family_history  2111 non-null   object 
 5   FAVC            2111 non-null   object 
 6   FCVC            2111 non-null   float64
 7   NCP             2111 non-null   float64
 8   CAEC            2111 non-null   object 
 9   SMOKE           2111 non-null   object 
 10  CH2O            2111 non-null   float64
 11  SCC             2111 non-null   object 
 12  FAF             2111 non-null   float64
 13  TUE             2111 non-null   float64
 14  CALC            2111 non-null   object 
 15  MTRANS          2111 non-null   object 
 16  Obesity         2111 non-null   object 
dtypes: float64(8), object(9)
memory u

In [ ]:
df['Obesity'].value_counts()

,count
Obesity,
Obesity_Type_I,351
Obesity_Type_III,324
Obesity_Type_II,297
Overweight_Level_I,290
Overweight_Level_II,290
Normal_Weight,287
Insufficient_Weight,272


In [ ]:
#Separando as independentes e a dependente

x = df.drop("Obesity", axis=1)
y = df["Obesity"]

In [ ]:
#Divisão de colunas numéricas e categóricas

num_features = [
    "Age", "Height", "Weight", "FCVC", "NCP",
    "CH2O", "FAF", "TUE"
]

cat_features = [
    "Gender", "family_history", "FAVC", "CAEC",
    "SMOKE", "SCC", "CALC", "MTRANS"
]


In [ ]:
#Feature Engineering: aplicando StandardScaler e OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
    ],
    remainder="passthrough"
)

In [ ]:
#Pipeline: LogisticRegression

pipeline_logreg = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])


In [ ]:
#Pipeline: GradientBoostingClassifier

pipeline_gb = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", GradientBoostingClassifier(random_state=42))
])


In [ ]:
#Pipeline: RandomForestClassifier

pipeline_rf = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(n_estimators=200, random_state=42))

])

In [ ]:
#Separando dados em treino: 80% e teste: 20%


x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
#Treinamento, previsões e cálculo da acurácia para cada modelo

pipelines = {
    "Logistic Regression": pipeline_logreg,
    "Gradient Boosting": pipeline_gb,
    "Random Forest": pipeline_rf
}

results = {}

for name, pipe in pipelines.items():
    pipe.fit(x_train, y_train)            # Treina a pipeline com o conjunto de treino
    y_pred = pipe.predict(x_test)         # Faz previsões no conjunto de teste
    acc = accuracy_score(y_test, y_pred)  # Calcula a acurácia
    results[name] = acc                   # Armazena no dicionário


In [ ]:
#Print da acurácia
#Apenas a acurácia já será suficinte para essa análise, pois os dados não estão com alto nível de desbalanceamento.

for model, acc in results.items():
    print(f"{model}: {acc:.4f}")


Logistic Regression: 0.8747
Gradient Boosting: 0.9504
Random Forest: 0.9433


#Será dado sequência com o Gradiente Boosting, já que alcançou características como:
Maior acurácia no conjunto de teste

Melhor capacidade de capturar padrões não lineares

Excelente para classificação multiclasse

In [ ]:
#Selecionando  e salvando o melhor modelo em disco para facilitar o deploy

best_model = pipelines["Gradient Boosting"]

joblib.dump(best_model, "modelo_obesidade.pkl")


['modelo_obesidade.pkl']

#Abaixo serão realizados os comandos para montarmos os arquivos essenciais ao app Streamplit:

###arquivo app.py

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

# Carregar dataset e modelo
df = pd.read_csv("Obesity.csv")
best_model = joblib.load("modelo_obesidade.pkl")

# Configuração da página
st.set_page_config(page_title="Sistema Preditivo de Obesidade", layout="wide")


# Criar abas
tab1, tab2, tab3, tab4 = st.tabs([
    "Problema de Negócio",
    "Análise Exploratória",
    "Sistema Preditivo",
    "Conclusões"
])



# ABA 1 — PROBLEMA DE NEGÓCIO
with tab1:
    st.header("Contexto e Problema de Negócio")

    st.markdown("""
    A obesidade é uma condição multifatorial que representa um importante problema de saúde pública.
    Ela está associada a diversas doenças crônicas, como diabetes, hipertensão, doenças cardiovasculares e outras complicações de saúde.
    Identificar precocemente o nível de obesidade de um indivíduo pode auxiliar profissionais de saúde a recomendar intervenções adequadas,
    prevenindo o agravamento de condições médicas.

    Este projeto faz parte do **Tech Challenge – Fase 4 da pós-graduação em Data Analytics** e tem como objetivo desenvolver um **sistema preditivo de obesidade**.
    A ferramenta foi criada para **auxiliar a equipe médica** na identificação do nível de obesidade de pacientes,
    usando dados demográficos, físicos e comportamentais de forma objetiva e baseada em dados.

    A base utilizada é o conjunto de dados **Obesity.csv**, que contém informações sobre idade, gênero, peso, altura, hábitos alimentares, nível de atividade física, histórico familiar e outros fatores que influenciam o desenvolvimento da obesidade.

    O problema é tratado como uma **classificação multiclasse**, pois o nível de obesidade é categorizado em diferentes classes:
    por exemplo, "Peso Normal", "Sobrepeso" e "Obesidade" (em diferentes níveis).

    O objetivo do sistema é fornecer **previsões precisas e rápidas**, apoiando decisões clínicas e contribuindo para ações preventivas e planejamento de tratamentos.
    """)



# ABA 2 — ANÁLISE EXPLORATÓRIA
with tab2:
    st.header("Análise Exploratória dos Dados")
    st.markdown("\n")

    # Criar duas colunas
    col1, col2 = st.columns(2)

    # ------------------- Coluna 1 -------------------
    with col1:
        # Distribuição das classes de obesidade
        st.markdown("### Distribuição das classes de obesidade")
        plt.figure(figsize=(4, 3))
        sns.countplot(
            data=df,
            x="Obesity",
            palette="pastel",
            order=df['Obesity'].value_counts().index
        )
        plt.title("Pacientes por obesidade", fontsize=10)
        plt.xlabel("Obesidade", fontsize=9)
        plt.ylabel("Contagem", fontsize=9)
        plt.xticks(rotation=30, fontsize=8)
        plt.yticks(fontsize=8)
        plt.tight_layout()
        st.pyplot(plt)

        st.markdown("\n")

        # Scatter plot: Idade x Peso
        st.markdown("### Idade x Peso")
        plt.figure(figsize=(4, 3))
        sns.scatterplot(
            data=df,
            x="Age",
            y="Weight",
            hue="Obesity",
            palette="bright",
            s=40
        )
        plt.title("Idade vs Peso", fontsize=10)
        plt.xlabel("Idade (anos)", fontsize=9)
        plt.ylabel("Peso (kg)", fontsize=9)
        plt.xticks(fontsize=8)
        plt.yticks(fontsize=8)
        plt.legend(fontsize=7, loc='upper right')
        plt.tight_layout()
        st.pyplot(plt)

    #Coluna 2
    with col2:
        # Obesidade x Frequência de Atividade Física (FAF)
        st.markdown("### Obesidade x Atividade Física")
        plt.figure(figsize=(4, 3))
        sns.boxplot(
            data=df,
            x="Obesity",
            y="FAF",
            palette="Set2"
        )
        plt.title("FAF por obesidade", fontsize=10)
        plt.xlabel("Obesidade", fontsize=9)
        plt.ylabel("Frequência de Atividade Física", fontsize=9)
        plt.xticks(rotation=30, fontsize=8)
        plt.yticks(fontsize=8)
        plt.tight_layout()
        st.pyplot(plt)

        st.markdown("\n")

        # Heatmap de correlação
        st.markdown("### Correlação entre variáveis numéricas")
        variaveis = ["Age", "Height", "Weight", "FCVC", "NCP", "CH2O", "FAF", "TUE"]
        dados_corr = df[variaveis]
        corr_matrix = dados_corr.corr()
        plt.figure(figsize=(5, 4))
        sns.heatmap(
            corr_matrix,
            annot=True,
            cmap="coolwarm",
            fmt=".2f",
            linewidths=0.5,
            cbar=True
        )
        plt.title("Correlação das variáveis", fontsize=10)
        plt.xticks(fontsize=8)
        plt.yticks(fontsize=8)
        plt.tight_layout()
        st.pyplot(plt)




# ABA 3 — SISTEMA PREDITIVO
with tab3:
    st.markdown("""
    <div style="
        background-color: #f0f2f6;
        padding: 10px;
        border-radius: 8px;
        box-shadow: 2px 2px 5px rgba(0,0,0,0.2);
    ">
        <h2 style="color: #0f4c81; font-weight: bold;">Sistema Preditivo de Obesidade</h2>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("""
    Preencha os dados do paciente abaixo e clique em **Prever Nível de Obesidade**.
    As opções apresentam **valores de referência** para facilitar a escolha.
    """)

    st.markdown("\n")

    with st.form("prediction_form"):
        col1, col2, col3 = st.columns(3)

        #Coluna 1 — Dados Pessoais
        with col1:
            st.subheader("Dados Pessoais")
            genero = st.selectbox("Gênero", ["Masculino", "Feminino"])
            idade = st.slider("Idade (anos)", 14, 61, 30)
            altura = st.slider("Altura (m)", 1.45, 1.98, 1.70)
            peso = st.slider("Peso (kg)", 39, 173, 70)

        #Coluna 2 — Hábitos Alimentares
        with col2:
            st.subheader("Hábitos Alimentares")
            historico_familiar = st.selectbox("Histórico familiar de obesidade", ["sim", "não"])
            consumo_calorico = st.selectbox("Consumo frequente de alimentos calóricos", ["sim", "não"])

            fcvc_opcoes = ["Raramente (1 vez/dia)", "Às vezes (2 vezes/dia)", "Sempre (3 vezes/dia)"]
            fcvc_cat = st.selectbox("Consumo de vegetais (FCVC)", fcvc_opcoes)
            fcvc = {"Raramente (1 vez/dia)": 1, "Às vezes (2 vezes/dia)": 2, "Sempre (3 vezes/dia)": 3}[fcvc_cat]

            ncp_opcoes = ["Uma (1 refeição)", "Duas (2 refeições)", "Três (3 refeições)", "Quatro ou mais (4+)"]
            ncp_cat = st.selectbox("Número de refeições (NCP)", ncp_opcoes)
            ncp = {"Uma (1 refeição)": 1, "Duas (2 refeições)": 2, "Três (3 refeições)": 3, "Quatro ou mais (4+)": 4}[ncp_cat]

            ch2o_opcoes = ["<1 L/dia", "1–2 L/dia", ">2 L/dia"]
            ch2o_cat = st.selectbox("Consumo diário de água (CH2O)", ch2o_opcoes)
            ch2o = {"<1 L/dia": 1, "1–2 L/dia": 2, ">2 L/dia": 3}[ch2o_cat]

        #Coluna 3 — Atividade e Estilo de Vida
        with col3:
            st.subheader("Atividade e Estilo de Vida")
            faf_opcoes = ["Nenhuma (0)", "1–2×/sem", "3–4×/sem", "5×/sem ou mais"]
            faf_cat = st.selectbox("Atividade física (FAF)", faf_opcoes)
            faf = {"Nenhuma (0)": 0, "1–2×/sem": 1, "3–4×/sem": 2, "5×/sem ou mais": 3}[faf_cat]

            tue_opcoes = ["0–2 h/dia", "3–5 h/dia", ">5 h/dia"]
            tue_cat = st.selectbox("Uso de tecnologia (TUE)", tue_opcoes)
            tue = {"0–2 h/dia": 0, "3–5 h/dia": 1, ">5 h/dia": 2}[tue_cat]

            calc = st.selectbox("Consumo de álcool", ["não", "Às vezes", "Frequentemente", "Sempre"])
            caec = st.selectbox("Alimentação entre refeições", ["não", "Às vezes", "Frequentemente", "Sempre"])
            fuma = st.selectbox("Fuma?", ["sim", "não"])
            monitora_calorias = st.selectbox("Monitora calorias?", ["sim", "não"])
            transporte = st.selectbox(
                "Meio de transporte",
                ["Automóvel", "Moto", "Bicicleta", "Transporte Público", "Caminhada"]
            )

        st.markdown("\n")
        botao = st.form_submit_button("Prever Nível de Obesidade")

        if botao:
            input_data = pd.DataFrame([{
                "Gender": genero,
                "Age": idade,
                "Height": altura,
                "Weight": peso,
                "family_history": historico_familiar,
                "FAVC": consumo_calorico,
                "FCVC": fcvc,
                "NCP": ncp,
                "CH2O": ch2o,
                "FAF": faf,
                "TUE": tue,
                "CALC": calc,
                "CAEC": caec,
                "SMOKE": fuma,
                "SCC": monitora_calorias,
                "MTRANS": transporte
            }])

            previsao = best_model.predict(input_data)[0]
            st.success(f"O nível de obesidade previsto é: **{previsao}**")





# ABA 4 — CONCLUSÕES
with tab4:
    st.header("Conclusões e Insights do Modelo")

    st.markdown("""
    - O modelo final escolhido foi o **Gradient Boosting Classifier**, por apresentar melhor desempenho na classificação multiclasse.
    - A acurácia do modelo atingiu mais de **75%**.
    - As variáveis mais relevantes foram:
        - Peso
        - Atividade física (FAF)
        - Hábitos alimentares (FAVC, FCVC, CAEC, CALC)
        - Histórico familiar (family_history)
        """)

Writing app.py


###Arquivo requirements.txt

In [ ]:
%%writefile requirements.txt
streamlit
pandas
numpy
scikit-learn==1.3.2
joblib==1.3.2
xgboost==2.0.3
matplotlib
seaborn

Writing requirements.txt


###Arquivo runtime.txt

In [ ]:
%%writefile runtime.txt
python-3.11

Writing runtime.txt


###Download para a máquina local dos arquivos que serão inseridos no GitHub

In [ ]:
from google.colab import files

files.download("app.py")
files.download("modelo_obesidade.pkl")
files.download("Obesity.csv")
files.download("requirements.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>